In [ ]:
import os
import glob
import random
import numpy as np
from PIL import Image
from PIL import ImageDraw
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from transformers import pipeline

from patches import get_image_patches

checkpoint = "google/owlv2-base-patch16-ensemble"
detector = pipeline(model=checkpoint, task="zero-shot-object-detection")

In [ ]:

files = os.path.join("..", "data", "artwork", "**")

patches = get_image_patches(files, batch_size=4096, patch_size=224, window_shift=56)

for _, _, X in patches:
    index = random.choices(range(X.shape[0]), k=1028)
    X = X[index, :]
    break

X.shape

In [ ]:
q = list(map(Image.fromarray, X))

predictions = []
images = []
for qi in tqdm(q):
    prediction = detector(
        qi,
        candidate_labels=["eye"],
    )
    if len(prediction):
        predictions.append(
            detector(
            qi,
            candidate_labels=["eye"],
            )
        )
        images.append(qi)

    if len(images) == 36:
        break

In [ ]:
drawings = []

for qi, prediction in zip(images, predictions):
    draw = ImageDraw.Draw(qi)

    for p in prediction:
        box = p["box"]
        label = p["label"]
        score = p["score"]

        xmin, ymin, xmax, ymax = box.values()
        draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=1)
        draw.text((xmin, ymin), f"{label}: {round(score,2)}", fill="white")

    drawings.append(qi)


f, ax = plt.subplots(ncols=6, nrows=6, figsize=(20, 20))
for i, img in enumerate(drawings):
    col = i // 6
    row = i % 6
    ax[col, row].imshow(np.array(img))
    ax[col, row].axes.get_xaxis().set_ticks([])
    ax[col, row].axes.get_yaxis().set_ticks([])